**Mount Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Install / Imports**

In [2]:
import pandas as pd
import numpy as np

**Load Data**

In [3]:
file_path = "/content/drive/MyDrive/c01-price-forecasting/data/raw/final_paddy_dataset.csv"
df = pd.read_csv(file_path)

print("Initial Shape:", df.shape)

Initial Shape: (2356, 21)


**Clean Column Names**

In [4]:
df.columns = df.columns.str.strip().str.lower()
print("Columns:", df.columns.tolist())

Columns: ['date', 'district', 'paddy_type', 'min_price', 'max_price', 'avg_price', 'production_total', 'season', 'price_range', 'week_of_year', 'week_sin', 'week_cos', 'price_t-1', 'price_t-2', 'price_t-3', 'price_t-4', 'price_t-8', 'price_t-12', 'price_4w_avg', 'price_8w_avg', 'price_change']


**Handle Date**

In [5]:
if 'date' not in df.columns:
    raise ValueError("'date' column missing")

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])

**Sorting**

In [6]:
if 'district' in df.columns:
    df = df.sort_values(by=['district', 'date'])
else:
    df = df.sort_values(by=['date'])

**Remove Duplicates + Missing Values**

In [7]:
df = df.drop_duplicates()

if 'district' in df.columns:
    df = df.sort_values(['district', 'date'])
    df = df.groupby('district', group_keys=False).apply(lambda x: x.ffill().bfill())
else:
    df = df.ffill().bfill()

/tmp/ipykernel_690/842859882.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('district', group_keys=False).apply(lambda x: x.ffill().bfill())


**Feature Engineering**

In [8]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week.astype('Int64')

def get_season(month):
    if month in [9,10,11,12,1,2,3]:
        return 'Maha'
    else:
        return 'Yala'

df['season'] = df['month'].apply(get_season)

**Lag + Rolling Features**

In [9]:
if 'avg_price' in df.columns:
    df['lag_1'] = df.groupby('district')['avg_price'].shift(1)
    df['lag_2'] = df.groupby('district')['avg_price'].shift(2)
    df['lag_4'] = df.groupby('district')['avg_price'].shift(4)
    df['lag_12'] = df.groupby('district')['avg_price'].shift(12)

df['rolling_mean_4'] = df.groupby('district')['avg_price'].rolling(4).mean().reset_index(level=0, drop=True)
df['rolling_std_4'] = df.groupby('district')['avg_price'].rolling(4).std().reset_index(level=0, drop=True)

df = df.dropna()

**Encoding**

In [10]:
df = pd.get_dummies(df, columns=['season'], drop_first=True)

**Save File**

In [11]:
output_path = "/content/drive/MyDrive/c01-price-forecasting/data/processed/cleaned_data.csv"
df.to_csv(output_path, index=False)

print("Preprocessing Completed!")
print("Final Shape:", df.shape)
df.head()

Preprocessing Completed!
Final Shape: (2308, 30)


,date,district,paddy_type,min_price,max_price,avg_price,production_total,price_range,week_of_year,week_sin,...,year,month,week,lag_1,lag_2,lag_4,lag_12,rolling_mean_4,rolling_std_4,season_Yala
48,2015-03-26,Ampara,Long_Grain_White,34.0,40.0,37.14,307661,6.0,13,1.000000,...,2015,3,13,37.48,38.50,36.34,27.50,37.6575,0.586195,False
53,2015-04-02,Ampara,Long_Grain_White,34.0,40.0,36.40,309335,6.0,14,0.992709,...,2015,4,14,37.14,37.48,37.51,26.10,37.3800,0.872238,True
59,2015-04-09,Ampara,Long_Grain_White,34.0,39.0,35.68,309335,5.0,15,0.970942,...,2015,4,15,36.40,37.14,38.50,26.58,36.6750,0.802060,True
63,2015-04-16,Ampara,Long_Grain_White,31.0,37.0,34.97,309335,6.0,16,0.935016,...,2015,4,16,35.68,36.40,37.48,28.95,36.0475,0.933430,True
64,2015-04-23,Ampara,Long_Grain_White,30.0,38.0,34.03,309335,8.0,17,0.885456,...,2015,4,17,34.97,35.68,37.14,30.64,35.2700,1.012028,True


**Download**

In [73]:
from google.colab import files
files.download("/content/drive/MyDrive/c01-price-forecasting/data/processed/cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nSample:")
print(df.head())

Shape: (2308, 30)

Columns: ['date', 'district', 'paddy_type', 'min_price', 'max_price', 'avg_price', 'production_total', 'price_range', 'week_of_year', 'week_sin', 'week_cos', 'price_t-1', 'price_t-2', 'price_t-3', 'price_t-4', 'price_t-8', 'price_t-12', 'price_4w_avg', 'price_8w_avg', 'price_change', 'year', 'month', 'week', 'lag_1', 'lag_2', 'lag_4', 'lag_12', 'rolling_mean_4', 'rolling_std_4', 'season_Yala']

Missing values:
0

Sample:
         date district        paddy_type  min_price  max_price  avg_price  \
48 2015-03-26   Ampara  Long_Grain_White       34.0       40.0      37.14   
53 2015-04-02   Ampara  Long_Grain_White       34.0       40.0      36.40   
59 2015-04-09   Ampara  Long_Grain_White       34.0       39.0      35.68   
63 2015-04-16   Ampara  Long_Grain_White       31.0       37.0      34.97   
64 2015-04-23   Ampara  Long_Grain_White       30.0       38.0      34.03   

    production_total  price_range  week_of_year  week_sin  ...  year  month  \
48            